In [0]:
import pyspark.sql.functions as f

%md
| Column | Description | Values / Meaning |
|---|---|---|
| `oph` | Operating hours of the engine | — |
| `pist_m` | Piston material | — |
| `issue_type` | Combustion issue type | `1` = typical issue<br>`2` = atypical issue<br>`3` = non-related<br>`4` = non-symptomatic |
| `bmep` | Brake mean effective pressure (average pressure forcing the pistons down) | — |
| `ng_imp` | Natural gas impurities (nmol) | — |
| `past_dmg` | Engine had past damages | `1` = true<br>`0` = false |
| `resting_analysis_results` | Resting results after operation | `0` = normal<br>`1` = abnormal<br>`2` = critical |
| `rpm_max` | Maximum rotations per minute achieved | — |
| `full_load_issues` | Issues induced due to full load operation | `1` = yes<br>`0` = no |
| `number_up` | Number of unplanned events | — |
| `number_tc` | Number of installed turbochargers | — |
| `op_set_1` | Operational engine setting 1 | — |
| `op_set_2` | Operational engine setting 2 | — |
| `op_set_3` | Operational engine setting 3 | — |
| `breakdown` | Breakdown probability indicator | `0` = less chance of breakdown<br>`1` = more chance of breakdown |

In [0]:
import pyspark.sql.types as t


schema = t.StructType(
    [
        t.StructField("engine_id", t.StringType(), True),
        t.StructField("timestamp", t.DateType(), True),
        t.StructField("oph", t.LongType(), True),
        t.StructField("pist_m", t.LongType(), True),
        t.StructField("issue_type", t.LongType(), True),
        t.StructField("bmep", t.DoubleType(), True),
        t.StructField("ng_imp", t.LongType(), True),
        t.StructField("past_dmg", t.BooleanType(), True),
        t.StructField("resting_analysis_results", t.LongType(), True),
        t.StructField("rpm_max", t.LongType(), True),
        t.StructField("full_load_issues", t.BooleanType(), True),
        t.StructField("number_up", t.DoubleType(), True),
        t.StructField("number_tc", t.DoubleType(), True),
        t.StructField("op_set_1", t.DoubleType(), True),
        t.StructField("op_set_2", t.DoubleType(), True),
        t.StructField("op_set_3", t.DoubleType(), True),
        t.StructField("breakdown", t.BooleanType(), True),
    ]
)

df = spark.read.table("workspace.case_study_test.01_input_data")

# Cast all columns at once to avoid deeply nested execution plans
casts = {field.name: f.col(field.name).cast(field.dataType) for field in schema.fields}
df = df.withColumns(casts)

df = df.filter(f.col("breakdown").isNotNull())

issue_type_map = {
    1: "typical issue",
    2: "atypical issue",
    3: "non-related",
    4: "non-symptomatic",
}

df = df.withColumn(
    "issue_type_desc",
    f.expr(
        "CASE issue_type "
        "WHEN 1 THEN 'typical issue' "
        "WHEN 2 THEN 'atypical issue' "
        "WHEN 3 THEN 'non-related' "
        "WHEN 4 THEN 'non-symptomatic' "
        "ELSE NULL END"
    ),
).withColumn("year_month", f.date_format(f.col("timestamp"), "yyyy-MM"))

df.display()

In [0]:
# Summary statistics for continuous variables by breakdown, key-value pairs in one column
from pyspark.sql import functions as f

cont_vars = [
    "oph",
    "bmep",
    "ng_imp",
    "rpm_max",
    "number_up",
    "number_tc",
    "op_set_1",
    "op_set_2",
    "op_set_3",
]

df_stat = df.groupBy("breakdown").count()

for var in cont_vars:

    df_var = df.groupBy("breakdown").agg(
        f.struct(
            f.round(f.mean(var), 2).alias(f"{var}_mean"),
            f.round(f.min(var).cast("double"), 2).alias(f"{var}_min"),
            f.round(f.max(var).cast("double"), 2).alias(f"{var}_max"),
            f.round(f.stddev(var), 2).alias(f"{var}_stddev"),
        ).alias(f"{var}")
    )
    df_stat = df_stat.join(df_var, ["breakdown"], "left")

df_stat.display()

In [0]:
df.groupBy(f.col("number_tc"), f.col("breakdown"), f.col("past_dmg")).count().display()

In [0]:
df.groupBy(
    f.col("breakdown"),
    f.col("number_tc"),
    f.col("issue_type_desc"),
    f.year(f.col("timestamp")).alias("timestamp"),
).count().display()

Databricks visualization. Run in Databricks to view.